# 04 — Train YOLOv8s on CUDA

Ports `image_rec_trainer.py`: `device='mps'` -> `device='cuda'`,
`yolov8n.pt` -> `yolov8s.pt` (larger/more accurate backbone, per your request).

The RTX 5060 Laptop GPU here has **8 GB VRAM**. `yolov8s` at `imgsz=640` fits
comfortably at moderate batch sizes with AMP (on by default in
`ultralytics`), but if you see a CUDA OOM: lower `BATCH`, or drop `IMGSZ` to
512, before reaching for a smaller model.


In [ ]:
import os
import random
import shutil
import yaml
from ultralytics import YOLO


## Config

In [ ]:
DATA_DIR = "nutri_grain_strawberry_bar_transparent_yolo_dataset"
MODEL_NAME = "yolov8s.pt"
RUN_NAME = "snacks"
IMGSZ = 640
BATCH = 16      # lower to 8 or 4 if you hit a CUDA OOM on the 8GB 5060
EPOCHS_FULL = 100


## Train/val auto-split + `data.yaml` path fix-up (unchanged from the original)

In [ ]:
def prepare_dataset(data_dir):
    images_dir = os.path.join(data_dir, "images")
    labels_dir = os.path.join(data_dir, "labels")
    yaml_path = os.path.join(data_dir, "data.yaml")
    train_img_dir = os.path.join(images_dir, "train")

    if not os.path.exists(train_img_dir):
        print("Train folder not found. Splitting data into train and val...")
        for folder in ["train", "val"]:
            os.makedirs(os.path.join(images_dir, folder), exist_ok=True)
            os.makedirs(os.path.join(labels_dir, folder), exist_ok=True)

        images = [
            f for f in os.listdir(images_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png")) and os.path.isfile(os.path.join(images_dir, f))
        ]
        random.seed(42)
        random.shuffle(images)
        split_idx = int(len(images) * 0.8)
        train_images, val_images = images[:split_idx], images[split_idx:]

        def move_files(file_list, dest_suffix):
            for img_name in file_list:
                shutil.move(os.path.join(images_dir, img_name), os.path.join(images_dir, dest_suffix, img_name))
                lbl_name = os.path.splitext(img_name)[0] + ".txt"
                lbl_src = os.path.join(labels_dir, lbl_name)
                if os.path.exists(lbl_src):
                    shutil.move(lbl_src, os.path.join(labels_dir, dest_suffix, lbl_name))

        move_files(train_images, "train")
        move_files(val_images, "val")
        print(f"Split {len(train_images)} train / {len(val_images)} val images.")
    else:
        print("Data already split into train/val. Skipping.")

    with open(yaml_path, "r") as f:
        data_config = yaml.safe_load(f)
    data_config["path"] = os.path.abspath(data_dir)
    data_config["train"] = "images/train"
    data_config["val"] = "images/val"
    with open(yaml_path, "w") as f:
        yaml.dump(data_config, f, sort_keys=False)
    print("data.yaml paths updated.")

    return yaml_path


## Train function

In [ ]:
def train_yolo_model(data_dir, model_name, run_name, epochs, imgsz, batch, device="cuda", script_dir=None):
    yaml_path = prepare_dataset(data_dir)

    print("Starting YOLO training...")
    model = YOLO(model_name)
    results = model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        name=run_name,
        device=device,
    )

    if script_dir and results and hasattr(results, "save_dir"):
        best_weights_path = os.path.join(results.save_dir, "weights", "best.pt")
        if os.path.exists(best_weights_path):
            destination_path = os.path.join(script_dir, f"{run_name}_best.pt")
            shutil.copy(best_weights_path, destination_path)
            print(f"Saved best weights to: {destination_path}")
        else:
            print("Training completed, but best.pt was not found.")

    return results


## Intermediary test — 1-epoch smoke run

Confirms the whole training loop (data loading, augmentation pipeline,
forward/backward pass, checkpoint save) runs end-to-end on the GPU without a
CUDA OOM, before committing to a full multi-epoch run. Uses the same dataset
and batch size as the full run so an OOM here means you need to shrink
`BATCH`/`IMGSZ` before running the real thing.


In [ ]:
SCRIPT_DIR = os.getcwd()

smoke_results = train_yolo_model(
    data_dir=DATA_DIR,
    model_name=MODEL_NAME,
    run_name=f"{RUN_NAME}_smoke",
    epochs=1,
    imgsz=IMGSZ,
    batch=BATCH,
    device="cuda",
    script_dir=None,
)
print("Smoke test finished OK — 1 epoch completed without CUDA OOM.")


## Full training run

Only run once the smoke test above finishes cleanly.

In [ ]:
RUN_FULL_TRAINING = False  # flip to True to run the real training

if RUN_FULL_TRAINING:
    results = train_yolo_model(
        data_dir=DATA_DIR,
        model_name=MODEL_NAME,
        run_name=RUN_NAME,
        epochs=EPOCHS_FULL,
        imgsz=IMGSZ,
        batch=BATCH,
        device="cuda",
        script_dir=SCRIPT_DIR,
    )


## Plot loss / mAP curves from the run

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_training_curves(save_dir):
    csv_path = os.path.join(save_dir, "results.csv")
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    loss_cols = [c for c in df.columns if "loss" in c]
    for c in loss_cols:
        axes[0].plot(df["epoch"], df[c], label=c)
    axes[0].set_title("loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend(fontsize=7)

    map_cols = [c for c in df.columns if "mAP" in c]
    for c in map_cols:
        axes[1].plot(df["epoch"], df[c], label=c)
    axes[1].set_title("mAP")
    axes[1].set_xlabel("epoch")
    axes[1].legend(fontsize=7)

    plt.tight_layout()
    plt.show()

if RUN_FULL_TRAINING:
    plot_training_curves(results.save_dir)
